# Diabetes Prediction Models

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [ ]:
df = pd.read_csv('dataset/diabetes.csv')
df.head()

## Preprocessing

General Information About the Dataset

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.columns.values

In [ ]:
df.groupby('Outcome').size()

## Visualizations

In [ ]:
category_counts = df['Outcome'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%')
plt.title('Pie Plot of Outcome')
plt.show()

In [ ]:
df.nunique()

In [ ]:
df.hist(bins=20, figsize=(20, 10))
plt.show()

In [ ]:
scatter_matrix(df, figsize=(30, 30), diagonal='hist', alpha=0.5)
plt.show()

## Data Cleaning

In [ ]:
print(df.isnull().sum())

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
plt.figure(figsize=(6, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Outlier Handling

Some columns use 0 as a placeholder for missing values. We replace these with the group median, then clip remaining outliers using IQR.

In [ ]:
print('total : ', df[df.BloodPressure == 0].shape[0])
df[df.BloodPressure == 0].groupby('Outcome')['Age'].count()

In [ ]:
print('total : ', df[df.Glucose == 0].shape[0])
df[df.Glucose == 0].groupby('Outcome')['Age'].count()

In [ ]:
print('total : ', df[df.SkinThickness == 0].shape[0])
df[df.SkinThickness == 0].groupby('Outcome')['Age'].count()

In [ ]:
print('total : ', df[df.BMI == 0].shape[0])
df[df.BMI == 0].groupby('Outcome')['Age'].count()

In [ ]:
print('total : ', df[df.Insulin == 0].shape[0])
df[df.Insulin == 0].groupby('Outcome')['Age'].count()

In [ ]:
# Replace impossible zeros with NaN then fill with group median
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_mod = df.copy()

for col in zero_cols:
    df_mod[col] = df_mod[col].replace(0, np.nan)
    df_mod[col] = df_mod.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

print(df_mod.shape)

In [ ]:
# Clip outliers using IQR — no rows dropped
numeric_cols = [c for c in df_mod.columns if c != 'Outcome']

for col in numeric_cols:
    Q1 = df_mod[col].quantile(0.25)
    Q3 = df_mod[col].quantile(0.75)
    IQR = Q3 - Q1
    df_mod[col] = df_mod[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print(df_mod.shape)

In [ ]:
# Before vs after box plots
fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(14, len(numeric_cols) * 3))

for i, col in enumerate(numeric_cols):
    axes[i, 0].boxplot(df[col].dropna(), vert=False)
    axes[i, 0].set_title(f'{col} - Before')
    axes[i, 1].boxplot(df_mod[col].dropna(), vert=False)
    axes[i, 1].set_title(f'{col} - After')

plt.suptitle('Outlier Treatment: Before vs After', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Standardization

In [ ]:
scaler = MinMaxScaler()
df_mod[numeric_cols] = scaler.fit_transform(df_mod[numeric_cols])
print(df_mod)

In [ ]:
df_mod.to_csv('cleaned_dataset.csv', index=False)

## Train / Test Split

In [ ]:
diabetes_df = pd.read_csv('cleaned_dataset.csv')

X = diabetes_df.drop('Outcome', axis=1)
y = diabetes_df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

accuracy_LR = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy_LR:.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

accuracy_DT = accuracy_score(y_test, y_pred)
print(f'Decision Tree Accuracy: {accuracy_DT:.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'Recall: {recall_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(dt, feature_names=X.columns, class_names=['No Diabetes', 'Diabetes'], filled=True, rounded=True)
plt.show()

## Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

accuracy_GB = accuracy_score(y_test, y_pred)
print(f'Gradient Boosting Accuracy: {accuracy_GB:.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'Recall: {recall_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - Gradient Boosting')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

accuracy_RF = accuracy_score(y_test, y_pred)
print(f'Random Forest Accuracy: {accuracy_RF:.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'Recall: {recall_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Support Vector Machine

In [ ]:
svm = SVC()
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

accuracy_SVM = accuracy_score(y_test, y_pred)
print(f'SVM Accuracy: {accuracy_SVM:.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'Recall: {recall_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - SVM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Neural Network

In [ ]:
nn = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=500, random_state=42)
nn.fit(X_train, y_train)
y_pred = nn.predict(X_test)

accuracy_NN = accuracy_score(y_test, y_pred)
print(f'Neural Network Accuracy: {accuracy_NN:.2f}')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=1):.2f}')
print(f'Recall: {recall_score(y_test, y_pred, zero_division=1):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred, zero_division=1):.2f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred, zero_division=1))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix - Neural Network')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Model Comparison

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Gradient Boosting', 'Random Forest', 'Support Vector Machine', 'Neural Network']
accuracy_scores = [accuracy_LR, accuracy_DT, accuracy_GB, accuracy_RF, accuracy_SVM, accuracy_NN]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracy_scores, color='blue')
plt.xlabel('Models')
plt.ylabel('Accuracy Score')
plt.title('Accuracy Scores of Different Models')
plt.ylim(0, 1)

for bar, accuracy in zip(bars, accuracy_scores):
    plt.text(bar.get_x() + bar.get_width() / 2 - 0.15, bar.get_height() + 0.02, f'{accuracy:.2f}', ha='center', color='black')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
all_models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Support Vector Machine': SVC(),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=500, random_state=42)
}

for name, model in all_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f'\n{name} Evaluation Metrics:')
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
    print(f'Classification Report:\n{classification_report(y_test, y_pred)}')

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                annot_kws={'size': 14},
                xticklabels=['No Diabetes', 'Diabetes'],
                yticklabels=['No Diabetes', 'Diabetes'])
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()